# Reliable Tool-Use SLM: QLoRA post-training with tool-decision supervision

Fine-tune Qwen2.5-1.5B-Instruct so it **calls tools when it should and does not when it shouldn't**.

Three checkpoints from the same base, so the only variable is the training data:

| Checkpoint | Data |
|---|---|
| **Base** | none |
| **Tool-SFT** | ~3K function-calling examples |
| **Reliable Tool-SFT** | ~3K function-calling + ~1K balanced When2Call decisions |

Code lives at [github.com/kushc2004/reliable-tool-use-slm](https://github.com/kushc2004/reliable-tool-use-slm)
and is cloned here, so there is one source of truth rather than a copy that drifts.

**Runtime:** ~2-3h on a T4. Well inside the 12h batch limit.

> **T4 note:** Turing has no native bfloat16. The trainer detects this via
> `torch.cuda.is_bf16_supported()` and falls back to fp16 automatically — requesting
> bf16 on a T4 does not raise, it silently degrades gradients.

## 1. Environment

In [ ]:
import os, sys, subprocess, json, shutil, time
import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__)
print('cuda    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device  :', torch.cuda.get_device_name(0))
    print('bf16 ok :', torch.cuda.is_bf16_supported())
    print('vram    : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## 2. Clone the repo

Pinned to a commit so a rerun reproduces the same code even if `main` moves.

In [ ]:
REPO = 'https://github.com/kushc2004/reliable-tool-use-slm.git'
PROJECT = '/kaggle/working/reliable-tool-use-slm'

if not os.path.exists(PROJECT):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, PROJECT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT, 'pull'], check=True)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('commit:', head)
print(subprocess.run(['ls'], capture_output=True, text=True).stdout)

## 3. Dependencies

Kaggle ships torch/transformers. bitsandbytes and peft may need installing.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'bitsandbytes>=0.43', 'peft>=0.11', 'accelerate>=0.33',
                'datasets>=2.20', 'PyYAML>=6.0'], check=True)

import importlib
for m in ['transformers', 'peft', 'bitsandbytes', 'datasets', 'accelerate']:
    mod = importlib.import_module(m)
    print('%-15s %s' % (m, getattr(mod, '__version__', '?')))

## 4. Sanity-check the scorer

Before trusting any model number, verify the measurement apparatus. A backend that
echoes the gold answer must score 100% on every decision metric. If it does not,
the scorer is broken and nothing downstream means anything.

This check caught a real bug during development, so it runs first here too.

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
assert r.returncode == 0, 'test suite failed — do not proceed'

## 5. Build the corpora

In [ ]:
N_TRAIN, N_W2C_TRAIN, N_EVAL = 3000, 1000, 250

# Positive tool-call corpus (Glaive function-calling v2).
!python -m src.data.build_dataset --source glaive --out data/processed --n-train $N_TRAIN --n-eval $N_EVAL --neg-ratio 0.0 --seed 0

In [ ]:
# When2Call eval set (the `mcq` split of the `test` config).
#
# The dataset is CONFIG-scoped, not split-scoped: train_sft and train_pref each
# live under their own config, and asking the default config for split="train"
# raises ValueError. prepare_when2call resolves the configs internally.
!python -m src.data.prepare_when2call --mode eval --out data/raw/w2c_eval.jsonl

# No separate training-subset step is needed: `build_dataset --source when2call`
# loads BOTH training configs itself. train_sft contributes only the non-call
# decisions (it contains no tool calls at all); train_pref contributes every
# tool_call row plus more non-call ones.

In [ ]:
# Reliable corpus: ~3K Glaive positives + When2Call decision supervision.
# neg_ratio 0.25 over 4000 rows gives 3000 positives + 1000 negatives.
!python -m src.data.build_dataset --source glaive,when2call --out data/reliable --n-train 4000 --n-eval $N_EVAL --neg-ratio 0.25 --seed 0

for split in ['processed', 'reliable']:
    p = 'data/%s/stats.json' % split
    if os.path.exists(p):
        s = json.load(open(p))
        print('\n=== %s ===' % split)
        print('  train:', s['train'])
        print('  eval :', s['eval'])
        print('  dropped:', s.get('dropped'))

## 6. Verify the oracle

Confirms the real When2Call test data scores 100% against a perfect backend.

In [ ]:
!python -m src.evaluate_when2call --data data/raw/w2c_eval.jsonl --backend oracle --out results/oracle_when2call

m = json.load(open('results/oracle_when2call/metrics.json'))
assert m['decision_accuracy'] == 1.0, 'ORACLE NOT 100%% — scorer is broken, stop here'
print('\nOracle verified at 100%%. Scorer is trustworthy.')

## 7. Train Tool-SFT

4-bit NF4 QLoRA, rank 16 / alpha 32, 3 epochs. fp16 on T4.

In [ ]:
t0 = time.time()
!python -m src.train_qlora --config configs/tool_sft.yaml --data data/processed --variant sft --out outputs/tool_sft
print('\nTool-SFT training took %.1f min' % ((time.time() - t0) / 60))

## 8. Train Reliable Tool-SFT

Same base model, trained **independently** rather than continued from Tool-SFT —
otherwise 'more data' and 'more training' are confounded.

In [ ]:
t0 = time.time()
!python -m src.train_qlora --config configs/reliable_tool_sft.yaml --data data/reliable --variant sft-neg --out outputs/reliable_tool_sft
print('\nReliable Tool-SFT training took %.1f min' % ((time.time() - t0) / 60))

## 9. Evaluate — tool-call track

In [ ]:
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'

!python -m src.evaluate --data data/processed --split all --checkpoint $BASE --out results/base
!python -m src.evaluate --data data/processed --split all --checkpoint $BASE --adapter outputs/tool_sft --out results/tool_sft
!python -m src.evaluate --data data/processed --split all --checkpoint $BASE --adapter outputs/reliable_tool_sft --out results/reliable_tool_sft

## 10. Evaluate — When2Call decision track

In [ ]:
!python -m src.evaluate_when2call --data data/raw/w2c_eval.jsonl --checkpoint $BASE --out results/base_when2call
!python -m src.evaluate_when2call --data data/raw/w2c_eval.jsonl --checkpoint $BASE --adapter outputs/tool_sft --out results/tool_sft_when2call
!python -m src.evaluate_when2call --data data/raw/w2c_eval.jsonl --checkpoint $BASE --adapter outputs/reliable_tool_sft --out results/reliable_tool_sft_when2call

## 11. Aggregate, analyse, plot

In [ ]:
# Flatten the per-run metrics into the names aggregate_results expects.
for run in ['base', 'tool_sft', 'reliable_tool_sft']:
    pairs = [('results/%s/metrics.json' % run, 'results/%s_metrics.json' % run),
             ('results/%s/failures.jsonl' % run, 'results/%s_failures.jsonl' % run),
             ('results/%s_when2call/metrics.json' % run, 'results/%s_when2call.json' % run),
             ('results/%s_when2call/failures.jsonl' % run, 'results/%s_when2call_failures.jsonl' % run)]
    for src, dst in pairs:
        if os.path.exists(src):
            shutil.copy(src, dst)

os.environ['MPLCONFIGDIR'] = '/kaggle/working/mpl'
!python -m src.aggregate_results --results results
!python -m src.error_analysis --results results --out results/error_analysis.json --max-examples 20
!python -m src.cv_metrics --results results --out results/cv_metrics.md

## 12. Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/cv_metrics.md').read()))

In [ ]:
for fig in ['exact_match.png', 'false_tool_call.png', 'confusion_matrix.png']:
    p = 'results/figures/%s' % fig
    if os.path.exists(p):
        display(Markdown('**%s**' % fig))
        display(Image(p))

## 13. Error analysis

In [ ]:
ea = json.load(open('results/error_analysis.json'))
print('Failure counts by category:')
for k, v in ea['counts'].items():
    print('  %-24s %d' % (k, v))

print('\n%d representative failures:' % ea['n_examples'])
for ex in ea['examples'][:5]:
    print('\n[%s/%s] %s' % (ex['checkpoint'], ex['track'], ex['category']))
    print('  %s' % ex['prediction'][:200])

## 14. Training provenance

Hardware, dtype, and parameter counts recorded at train time — so a result can be
traced back to the conditions that produced it.

In [ ]:
for run in ['tool_sft', 'reliable_tool_sft']:
    p = 'outputs/%s/run_config.json' % run
    if os.path.exists(p):
        m = json.load(open(p))
        print('=== %s ===' % run)
        for k in ['n_records', 'trainable_params', 'total_params', 'trainable_pct',
                  'compute_dtype', 'device', 'peak_gpu_mem_gb']:
            print('  %-18s %s' % (k, m.get(k)))
        print()

In [ ]:
# Bundle every artifact so the notebook output is self-contained.
shutil.make_archive('/kaggle/working/reliable_tool_use_results', 'zip', 'results')
print('wrote /kaggle/working/reliable_tool_use_results.zip')
print(subprocess.run(['du', '-sh', 'results', 'outputs'], capture_output=True, text=True).stdout)